In [3]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import warnings
warnings.filterwarnings('ignore')

# ==========================================
# 1. RAW DATA TO MONTHLY AGGREGATION
# ==========================================
print("1. Processing Raw Data directly to Monthly Level...")

data_claim = pd.read_csv('data/Data_Klaim.csv')
data_claim['Tanggal Pasien Masuk RS'] = pd.to_datetime(data_claim['Tanggal Pasien Masuk RS'])

valid_claims = data_claim[data_claim['Status Klaim'] == 'PAID'].copy()
valid_claims.set_index('Tanggal Pasien Masuk RS', inplace=True)

# Resample to Month-End ('ME')
monthly_df = valid_claims.resample('ME').agg(
    Frequency=('Nomor Polis', 'count'),
    Total_Claim=('Nominal Klaim Yang Disetujui', 'sum')
).reset_index()

monthly_df.rename(columns={'Tanggal Pasien Masuk RS': 'Month_End_Date'}, inplace=True)
monthly_df['Month_Period'] = monthly_df['Month_End_Date'].dt.to_period('M')

# ==========================================
# 2. FEATURE ENGINEERING (LAGS)
# ==========================================
print("2. Generating Lag Features for Tree Modeling...")

# We only look back 3 months. If we look back further, we burn too much training data.
lookback = 3    

targets = ['Frequency', 'Total_Claim']

for col in targets:
    for lag in range(1, lookback + 1):
        monthly_df[f'{col}_Lag{lag}'] = monthly_df[col].shift(lag)

train_df = monthly_df.dropna().copy()
print(f"Training on only {len(train_df)} months of data. LightGBM will need special parameters!")

# ==========================================
# 3. TRAIN LIGHTGBM (SMALL-DATA TUNED)
# ==========================================
print("3. Training LightGBM Trees...")

# >>> THE SMALL DATA UPGRADES <<<
# Standard LightGBM needs 20 samples to make a tree leaf. We only have ~16 rows total!
# We MUST drop min_child_samples and limit the tree depth so it doesn't overfit to noise.
lgb_params = {
    'n_estimators': 80,          # Don't build too many trees
    'num_leaves': 5,             # Keep the trees extremely shallow/simple
    'min_child_samples': 2,      # FORCE it to learn even with tiny data slices
    'random_state': 42,
    'verbose': -1
}

# --- FREQUENCY MODEL ---
features_freq = [f'Frequency_Lag{i}' for i in range(1, lookback + 1)]
model_lgb_freq = lgb.LGBMRegressor(**lgb_params)
model_lgb_freq.fit(train_df[features_freq], train_df['Frequency'])

# --- TOTAL CLAIM MODEL ---
features_tot = [f'Total_Claim_Lag{i}' for i in range(1, lookback + 1)]
model_lgb_tot = lgb.LGBMRegressor(**lgb_params)
model_lgb_tot.fit(train_df[features_tot], train_df['Total_Claim'])

# ==========================================
# 4. RECURSIVE MONTHLY FORECASTING
# ==========================================
print("4. Forecasting August 2025 to December 2025...")

target_months = pd.period_range(start='2025-08', end='2025-12', freq='M')

current_history = monthly_df.copy()
monthly_predictions = []

for target_m in target_months:
    last_n = current_history.tail(lookback)
    
    # Extract lags for prediction
    row_freq = pd.DataFrame([{f'Frequency_Lag{i}': last_n['Frequency'].iloc[-i] for i in range(1, lookback + 1)}])
    row_tot = pd.DataFrame([{f'Total_Claim_Lag{i}': last_n['Total_Claim'].iloc[-i] for i in range(1, lookback + 1)}])
    
    # Predict
    pred_freq = max(0, model_lgb_freq.predict(row_freq)[0])
    pred_tot = max(0, model_lgb_tot.predict(row_tot)[0])
    
    # Store Prediction
    monthly_predictions.append({
        'Month_Period': target_m,
        'Frequency': int(np.round(pred_freq)),
        'Total_Claim': pred_tot
    })
    
    # Update History so the next loop can use this prediction as a lag
    new_row_data = {
        'Month_End_Date': target_m.to_timestamp(how='end'), 
        'Month_Period': target_m,
        'Frequency': pred_freq, 
        'Total_Claim': pred_tot
    }
    
    current_history = pd.concat([current_history, pd.DataFrame([new_row_data])], ignore_index=True)

# ==========================================
# 5. FORMAT AND EXPORT
# ==========================================
print("5. Formatting for Kaggle Submission...")

final_forecast = pd.DataFrame(monthly_predictions)
final_forecast['Severity'] = np.where(final_forecast['Frequency'] > 0, 
                                      final_forecast['Total_Claim'] / final_forecast['Frequency'], 
                                      0)

formatted_data = []
for index, row in final_forecast.iterrows():
    m_id = str(row['Month_Period']).replace('-', '_')
    formatted_data.append({'id': f"{m_id}_Claim_Frequency", 'value': row['Frequency']})
    formatted_data.append({'id': f"{m_id}_Claim_Severity", 'value': row['Severity']})
    formatted_data.append({'id': f"{m_id}_Total_Claim", 'value': row['Total_Claim']})

submission_df = pd.DataFrame(formatted_data)
submission_df.to_csv('submission_lgbm_monthly.csv', index=False)

print("\n--- FINAL FORECAST (Monthly LightGBM) ---")
print(final_forecast[['Month_Period', 'Frequency', 'Severity', 'Total_Claim']])
print("\nFile 'submission_lgbm_monthly.csv' saved!")

1. Processing Raw Data directly to Monthly Level...
2. Generating Lag Features for Tree Modeling...
Training on only 16 months of data. LightGBM will need special parameters!
3. Training LightGBM Trees...
4. Forecasting August 2025 to December 2025...
5. Formatting for Kaggle Submission...

--- FINAL FORECAST (Monthly LightGBM) ---
  Month_Period  Frequency      Severity   Total_Claim
0      2025-08        228  5.338785e+07  1.217243e+10
1      2025-09        250  3.866282e+07  9.665705e+09
2      2025-10        228  7.550991e+07  1.721626e+10
3      2025-11        210  6.431599e+07  1.350636e+10
4      2025-12        267  4.297230e+07  1.147360e+10

File 'submission_lgbm_monthly.csv' saved!
